<a href="https://colab.research.google.com/github/ClarenceBallensky/MLP-vs-GCN-ids/blob/main/MLP_vs_GCN_Intrustion_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLP v.s. GCN for Intrusion Detection
#### Clarence Ballensky
Research Question: Can a Graph Convolutional Network (GCN) improve network intrusion detection by leveraging relationships between network flows, compared with a Multilayer Perceptron (MLP), which treats each flow independently?

In [1]:
import os
from google.colab import drive
import zipfile, pandas as pd

In [2]:
drive.mount('/content/drive')

URL = "https://intrusion-detection.distrinet-research.be/CNS2022/Datasets/CSECICIDS2018_improved.zip"
FRAC = 0.10
OUT = "/content/drive/MyDrive/ids_samples"
os.makedirs(OUT, exist_ok=True)

!wget -q -c -O /content/data.zip {URL}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
with zipfile.ZipFile("/content/data.zip") as z:
    for name in z.namelist():
        out = f"{OUT}/{os.path.basename(name)}.parquet"
        if name.endswith(".csv") and not os.path.exists(out):
            chunks = pd.read_csv(z.open(name), chunksize=200_000)
            sample = [c.dropna().sample(frac=FRAC, random_state=42) for c in chunks]
            pd.concat(sample).to_parquet(out)
            print("done:", name)

done: Wednesday-14-02-2018.csv
done: Friday-16-02-2018.csv
done: Tuesday-20-02-2018.csv
done: Wednesday-21-02-2018.csv
done: Thursday-22-02-2018.csv
done: Friday-23-02-2018.csv
done: Wednesday-28-02-2018.csv
done: Thursday-01-03-2018.csv
done: Thursday-15-02-2018.csv
done: Friday-02-03-2018.csv


In [ ]:
df = pd.concat(pd.read_parquet(f"{OUT}/{f}") for f in os.listdir(OUT) if f.endswith(".parquet"))
print(df.shape)

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!ls -lh /content/drive/MyDrive/ids_samples

Mounted at /content/drive
total 1.3G
-rw------- 1 root root 130M Nov 11 21:50 Friday-02-03-2018.csv.parquet
-rw------- 1 root root 137M Nov 11 21:41 Friday-16-02-2018.csv.parquet
-rw------- 1 root root 122M Nov 11 21:45 Friday-23-02-2018.csv.parquet
-rw------- 1 root root 135M Nov 11 21:48 Thursday-01-03-2018.csv.parquet
-rw------- 1 root root 106M Nov 11 21:49 Thursday-15-02-2018.csv.parquet
-rw------- 1 root root 124M Nov 11 21:44 Thursday-22-02-2018.csv.parquet
-rw------- 1 root root 120M Nov 11 21:42 Tuesday-20-02-2018.csv.parquet
-rw------- 1 root root 113M Nov 11 21:40 Wednesday-14-02-2018.csv.parquet
-rw------- 1 root root 134M Nov 11 21:43 Wednesday-21-02-2018.csv.parquet
-rw------- 1 root root 134M Nov 11 21:47 Wednesday-28-02-2018.csv.parquet


In [2]:
import os, hashlib, pandas as pd
import pyarrow.parquet as pq

OUT = "/content/drive/MyDrive/ids_samples"
TARGET_ROWS = 300_000
SEED = 42

files = sorted(f for f in os.listdir(OUT) if f.endswith(".parquet"))
total = sum(pq.ParquetFile(f"{OUT}/{f}").metadata.num_rows for f in files)
frac = min(1.0, TARGET_ROWS / total)
print(total, "rows available; keeping fraction", round(frac, 3))

df = pd.concat(
    pd.read_parquet(f"{OUT}/{f}").sample(frac=frac, random_state=SEED)
    for f in files
).reset_index(drop=True)

fl = df.select_dtypes("float64").columns
df[fl] = df[fl].astype("float32")            # halves memory for float columns

df.to_parquet("/content/ids_sample.parquet")
print(df.shape)
print(os.path.getsize("/content/ids_sample.parquet") / 1e6, "MB")
print("sha256:", hashlib.sha256(open("/content/ids_sample.parquet", "rb").read()).hexdigest())

6319514 rows available; keeping fraction 0.047
(300001, 91)
56.817488 MB
sha256: f4eb850ee32419da5ab3aee9b33da28b1b3a3ab175c7e7af64a4311163fb03d5


In [3]:
from google.colab import files
files.download("/content/ids_sample.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os, hashlib, urllib.request
import numpy as np, pandas as pd

SAMPLE_URL = "https://github.com/ClarenceBallensky/MLP-vs-GCN-ids/releases/download/v1/ids_sample.parquet"
SAMPLE_PATH = "ids_sample.parquet"
EXPECTED_SHA256 = "f4eb850ee32419da5ab3aee9b33da28b1b3a3ab175c7e7af64a4311163fb03d5"
SEED = 42
np.random.seed(SEED)

if not os.path.exists(SAMPLE_PATH):
    urllib.request.urlretrieve(SAMPLE_URL, SAMPLE_PATH)

with open(SAMPLE_PATH, "rb") as f:
    assert hashlib.sha256(f.read()).hexdigest() == EXPECTED_SHA256, "Dataset file is corrupt or changed"

df = pd.read_parquet(SAMPLE_PATH)
print(df.shape)